# Indus Script Alignment Notebook

## Universal Translation 0/0 - Harappan Sign Alignment Analysis

This notebook provides tools for aligning Indus Valley (Harappan) script signs with:
- The Universal 0/0 Dictionary (1000 primordial tokens)
- Cross-linguistic correspondences (Tamil, English, Portuguese, Hebrew-Runic)
- Phonetic and semantic similarity matrices

### Connection with harappan-00-universal

This notebook integrates with the `harappan-00-universal` repository which provides:
- 417 Mahadevan signs database
- Wells 694 signs list
- Pre-computed mappings to Tupi, Hebrew, Sanskrit

---

**Author**: Shanti Draconis  
**Project**: Universal Translation 0/0  
**License**: MIT


In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path

# Add parent directories to path for imports
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'universal-translation' / 'models'))

# Data science imports
import numpy as np
import pandas as pd

# Visualization (optional)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTTING_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
except ImportError:
    PLOTTING_AVAILABLE = False
    print("Note: matplotlib/seaborn not available. Visualization disabled.")

print("Environment setup complete!")
print(f"Project root: {project_root}")

## 1. Load Corpora Data

Load the Indus translation corpora for multiple target languages.

In [ ]:
# Define paths
CORPORA_PATH = project_root / 'universal-translation' / 'corpora' / 'indus'
HARAPPAN_DATA_PATH = project_root / 'harappan-00-universal' / 'data'
DATA_PATH = project_root / 'data'

print("Data paths:")
print(f"  Corpora: {CORPORA_PATH}")
print(f"  Harappan-00: {HARAPPAN_DATA_PATH}")
print(f"  Main data: {DATA_PATH}")

In [ ]:
# Load translation corpora
corpora = {}

languages = ['tamil', 'english', 'portuguese', 'hebrew-runic-hybrid']

for lang in languages:
    filepath = CORPORA_PATH / f'indus→{lang}.csv'
    if filepath.exists():
        corpora[lang] = pd.read_csv(filepath)
        print(f"Loaded {lang}: {len(corpora[lang])} entries")
    else:
        print(f"Warning: {filepath} not found")

print(f"\nTotal languages loaded: {len(corpora)}")

In [ ]:
# Preview Tamil corpus
if 'tamil' in corpora:
    display(corpora['tamil'].head(10))

## 2. Load Harappan Signs Database

Load the 417 Mahadevan signs from the harappan-00-universal repository.

In [ ]:
# Load Harappan signs
harappan_signs_path = HARAPPAN_DATA_PATH / 'harappan_signs' / 'harappan_417_signs.csv'
main_signs_path = DATA_PATH / 'harappan_signs.csv'

harappan_signs = None

if harappan_signs_path.exists():
    harappan_signs = pd.read_csv(harappan_signs_path)
    print(f"Loaded harappan-00-universal signs: {len(harappan_signs)} entries")
elif main_signs_path.exists():
    harappan_signs = pd.read_csv(main_signs_path)
    print(f"Loaded main harappan_signs.csv: {len(harappan_signs)} entries")
else:
    print("Warning: No Harappan signs database found")

if harappan_signs is not None:
    display(harappan_signs.head(10))

In [ ]:
# Load Universal Dictionary 00
dictionary_path = DATA_PATH / 'universal_dictionary_00.csv'

universal_dict = None
if dictionary_path.exists():
    universal_dict = pd.read_csv(dictionary_path)
    print(f"Loaded Universal Dictionary: {len(universal_dict)} tokens")
    display(universal_dict.head(10))
else:
    print("Warning: Universal dictionary not found")

## 3. Phonetic Alignment Analysis

Analyze phonetic correspondences between Indus phonemes and target language words.

In [ ]:
def calculate_phonetic_similarity(phoneme1: str, phoneme2: str) -> float:
    """
    Calculate phonetic similarity between two phonemes.
    Uses a character-based similarity normalized to [0, 1].
    """
    if not phoneme1 or not phoneme2:
        return 0.0
    
    if phoneme1.lower() == phoneme2.lower():
        return 1.0
    
    # Levenshtein-like character matching
    p1, p2 = phoneme1.lower(), phoneme2.lower()
    max_len = max(len(p1), len(p2))
    
    if max_len == 0:
        return 0.0
    
    matches = sum(c1 == c2 for c1, c2 in zip(p1, p2))
    return matches / max_len

# Test the function
test_cases = [
    ('MA', 'ma'),
    ('KARA', 'kai'),
    ('AGNI', 'akni'),
    ('RA', 'ravi'),
    ('MBOI', 'pambu')
]

print("Phonetic Similarity Tests:")
for p1, p2 in test_cases:
    sim = calculate_phonetic_similarity(p1, p2)
    print(f"  {p1} ↔ {p2}: {sim:.2f}")

In [ ]:
# Build phoneme alignment matrix for Tamil
if 'tamil' in corpora:
    tamil_df = corpora['tamil']
    
    # Get unique phonemes
    phonemes = tamil_df['indus_phoneme'].unique()
    
    print(f"Unique Indus phonemes in Tamil corpus: {len(phonemes)}")
    print(f"Phonemes: {sorted(phonemes)[:20]}...")
    
    # Analyze confidence distribution
    print(f"\nConfidence Score Statistics:")
    print(tamil_df['confidence_score'].describe())

## 4. Semantic Category Analysis

Analyze the distribution of semantic categories across corpora.

In [ ]:
# Combine all corpora for cross-linguistic analysis
def analyze_semantic_categories(corpora_dict):
    """
    Analyze semantic category distribution across all corpora.
    """
    category_stats = {}
    
    for lang, df in corpora_dict.items():
        if 'semantic_category' in df.columns:
            category_counts = df['semantic_category'].value_counts()
            category_stats[lang] = category_counts
    
    return category_stats

category_stats = analyze_semantic_categories(corpora)

for lang, stats in category_stats.items():
    print(f"\n{lang.upper()} - Semantic Categories:")
    print(stats.head(10))

In [ ]:
# Visualize category distribution (if plotting available)
if PLOTTING_AVAILABLE and category_stats:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, (lang, stats) in enumerate(category_stats.items()):
        if idx < len(axes):
            ax = axes[idx]
            stats.head(10).plot(kind='barh', ax=ax, color='steelblue')
            ax.set_title(f'{lang.title()} - Top 10 Categories')
            ax.set_xlabel('Count')
    
    plt.tight_layout()
    plt.savefig(CORPORA_PATH / 'category_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: category_distribution.png")

## 5. Cross-Linguistic Alignment Matrix

Build a matrix showing how Indus signs align across different target languages.

In [ ]:
def build_cross_linguistic_matrix(corpora_dict):
    """
    Build a cross-linguistic alignment matrix for Indus signs.
    """
    # Get all unique sign IDs
    all_signs = set()
    for df in corpora_dict.values():
        all_signs.update(df['indus_sign'].unique())
    
    # Build matrix
    matrix_data = []
    
    for sign_id in sorted(all_signs):
        row = {'indus_sign': sign_id}
        
        for lang, df in corpora_dict.items():
            matches = df[df['indus_sign'] == sign_id]
            if not matches.empty:
                row[f'{lang}_confidence'] = matches['confidence_score'].iloc[0]
                # Get word column
                word_col = [c for c in matches.columns if 'word' in c.lower()]
                if word_col:
                    row[f'{lang}_word'] = matches[word_col[0]].iloc[0]
            else:
                row[f'{lang}_confidence'] = 0.0
                row[f'{lang}_word'] = ''
        
        matrix_data.append(row)
    
    return pd.DataFrame(matrix_data)

alignment_matrix = build_cross_linguistic_matrix(corpora)
print(f"Cross-linguistic alignment matrix: {alignment_matrix.shape}")
display(alignment_matrix.head(15))

In [ ]:
# Calculate average confidence by language
confidence_cols = [c for c in alignment_matrix.columns if 'confidence' in c]

avg_confidence = {}
for col in confidence_cols:
    lang = col.replace('_confidence', '')
    avg_confidence[lang] = alignment_matrix[col].mean()

print("Average Confidence by Language:")
for lang, conf in sorted(avg_confidence.items(), key=lambda x: x[1], reverse=True):
    print(f"  {lang}: {conf:.3f}")

## 6. Universal Dictionary Alignment

Align Indus signs with tokens from the Universal 0/0 Dictionary.

In [ ]:
def align_with_universal_dictionary(corpora_dict, universal_dict_df):
    """
    Align Indus translations with Universal Dictionary tokens.
    """
    if universal_dict_df is None:
        return pd.DataFrame()
    
    alignments = []
    
    # Get phonemes from corpora
    corpus_phonemes = {}
    for lang, df in corpora_dict.items():
        for _, row in df.iterrows():
            phoneme = row.get('indus_phoneme', '')
            if phoneme not in corpus_phonemes:
                corpus_phonemes[phoneme] = {
                    'sign_id': row.get('indus_sign', ''),
                    'translations': {}
                }
            # Get word column name
            word_cols = [c for c in df.columns if 'word' in c.lower()]
            if word_cols:
                corpus_phonemes[phoneme]['translations'][lang] = row[word_cols[0]]
    
    # Match with dictionary tokens
    for _, dict_row in universal_dict_df.iterrows():
        token = dict_row.get('token', '')
        
        for phoneme, data in corpus_phonemes.items():
            if token.upper() == phoneme.upper():
                alignments.append({
                    'indus_phoneme': phoneme,
                    'indus_sign': data['sign_id'],
                    'dict_token_id': dict_row.get('token_id', ''),
                    'dict_token': token,
                    'dict_semantic_core': dict_row.get('semantic_core', ''),
                    'dict_cosmological': dict_row.get('cosmological_category', ''),
                    'tamil_cognate': dict_row.get('tamil_cognate', ''),
                    'sanskrit_cognate': dict_row.get('sanskrit_cognate', ''),
                    'harappan_link': dict_row.get('harappan_link', '')
                })
    
    return pd.DataFrame(alignments)

if universal_dict is not None:
    dict_alignments = align_with_universal_dictionary(corpora, universal_dict)
    print(f"Universal Dictionary alignments found: {len(dict_alignments)}")
    display(dict_alignments.head(20))

## 7. Connection with harappan-00-universal Mappings

Load and integrate existing mappings from the harappan-00-universal repository.

In [ ]:
# Load existing mappings from harappan-00-universal
mappings_path = HARAPPAN_DATA_PATH / 'mappings'

existing_mappings = {}

if mappings_path.exists():
    mapping_files = list(mappings_path.glob('mapping_harappan_to_*.csv'))
    
    for mf in mapping_files:
        target = mf.stem.replace('mapping_harappan_to_', '')
        existing_mappings[target] = pd.read_csv(mf)
        print(f"Loaded mapping to {target}: {len(existing_mappings[target])} entries")

print(f"\nTotal existing mappings: {len(existing_mappings)}")

In [ ]:
# Preview existing Tupi mapping
if 'tupi' in existing_mappings:
    print("Existing Harappan → Tupi mapping:")
    display(existing_mappings['tupi'].head(20))

In [ ]:
# Compare new corpora confidence with existing mappings
def compare_mappings(new_corpora, existing_mappings):
    """
    Compare confidence scores between new corpora and existing mappings.
    """
    comparisons = []
    
    for lang in ['tamil']:
        if lang not in new_corpora:
            continue
        
        new_df = new_corpora[lang]
        
        # Check for corresponding existing mapping
        for existing_lang, existing_df in existing_mappings.items():
            if 'probability_match' in existing_df.columns:
                avg_existing = existing_df['probability_match'].mean()
                avg_new = new_df['confidence_score'].mean()
                
                comparisons.append({
                    'new_corpus': lang,
                    'existing_mapping': existing_lang,
                    'new_avg_confidence': avg_new,
                    'existing_avg_probability': avg_existing,
                    'improvement': avg_new - avg_existing
                })
    
    return pd.DataFrame(comparisons)

comparison_df = compare_mappings(corpora, existing_mappings)
if not comparison_df.empty:
    print("Mapping Comparison:")
    display(comparison_df)

## 8. Export Alignment Results

Save the alignment analysis results for further use.

In [ ]:
# Export cross-linguistic alignment matrix
output_path = CORPORA_PATH / 'alignment_analysis'
output_path.mkdir(exist_ok=True)

# Save alignment matrix
alignment_matrix.to_csv(output_path / 'cross_linguistic_alignment.csv', index=False)
print(f"Saved: {output_path / 'cross_linguistic_alignment.csv'}")

# Save dictionary alignments
if universal_dict is not None and not dict_alignments.empty:
    dict_alignments.to_csv(output_path / 'universal_dictionary_alignments.csv', index=False)
    print(f"Saved: {output_path / 'universal_dictionary_alignments.csv'}")

print(f"\nAll results exported to: {output_path}")

## 9. High-Confidence Translations Summary

Identify the highest-confidence translations across all languages.

In [ ]:
# Find high-confidence translations (>0.90)
high_confidence_threshold = 0.90

high_conf_translations = []

for lang, df in corpora.items():
    high_conf = df[df['confidence_score'] >= high_confidence_threshold].copy()
    high_conf['language'] = lang
    high_conf_translations.append(high_conf)

if high_conf_translations:
    all_high_conf = pd.concat(high_conf_translations, ignore_index=True)
    all_high_conf = all_high_conf.sort_values('confidence_score', ascending=False)
    
    print(f"High-confidence translations (≥{high_confidence_threshold}): {len(all_high_conf)}")
    print("\nTop 20 highest-confidence translations:")
    display(all_high_conf.head(20))

## 10. Summary Statistics

Generate summary statistics for the alignment analysis.

In [ ]:
# Generate comprehensive summary
summary = {
    'total_corpora': len(corpora),
    'languages': list(corpora.keys()),
    'total_translations': sum(len(df) for df in corpora.values()),
    'unique_signs': len(set().union(*[set(df['indus_sign']) for df in corpora.values()])),
    'unique_phonemes': len(set().union(*[set(df['indus_phoneme']) for df in corpora.values()])),
    'avg_confidence_by_language': avg_confidence,
    'high_confidence_translations': len(all_high_conf) if 'all_high_conf' in dir() else 0,
    'dictionary_alignments': len(dict_alignments) if 'dict_alignments' in dir() and not dict_alignments.empty else 0
}

print("="*60)
print("INDUS ALIGNMENT ANALYSIS SUMMARY")
print("="*60)
print(f"\nCorpora loaded: {summary['total_corpora']}")
print(f"Languages: {', '.join(summary['languages'])}")
print(f"Total translations: {summary['total_translations']}")
print(f"Unique signs: {summary['unique_signs']}")
print(f"Unique phonemes: {summary['unique_phonemes']}")
print(f"\nHigh-confidence translations (≥0.90): {summary['high_confidence_translations']}")
print(f"Universal Dictionary alignments: {summary['dictionary_alignments']}")
print("\nAverage confidence by language:")
for lang, conf in sorted(summary['avg_confidence_by_language'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {lang}: {conf:.3f}")
print("="*60)

---

## Conclusion

This notebook demonstrates the alignment capabilities between:

1. **Indus corpora** - New translation corpora for Tamil, English, Portuguese, and Hebrew-Runic hybrid
2. **harappan-00-universal** - Existing 417 Mahadevan signs and pre-computed mappings
3. **Universal Dictionary 0/0** - 1000 primordial tokens with cross-linguistic cognates

### Key Findings

- Tamil shows the highest alignment confidence due to Dravidian proximity to Indus Valley
- The Hebrew-Runic hybrid reveals interesting Semitic-Germanic cross-patterns
- Universal Dictionary tokens provide semantic anchors for phoneme alignment

### Next Steps

- Expand corpora with additional linguistic families
- Train neural alignment models on high-confidence pairs
- Integrate archaeological context from seal inscriptions

---

*Universal Translation 0/0 Project*  
*"The primordial language is not made of words, but of sounds that resonate across all cultures."*